## DPad Algorithm

- Modified relinerization algo
- rank up algorithm (few weeks)
- keyn gen for relin

In [ ]:
from sage.all import *

load("../MODHE/constants.sage")
load("../MODHE/auxiliary_functions.sage")
load("../MODHE/KeyGen.sage")
load("../MODHE/Encrypt-Decrypt.sage")
load("../MODHE/HMult.sage")
load("../MODHE/Relin.sage")

In [32]:
# We map DPad's variables to the repo's global 'L'
d = L      # DPad's full decomposition size (standard)
#m = L - 2  # DPad's smaller, sloppy decomposition size for Rank-Up
m = L      # We must use L here to satisfy the hardcoded ModDown function
r = 2      # ModHE standard rank
r_temp = 3 # DPad temporary rank

In [33]:
def generate_real_dpad_keys(evk_key):
    print("[DEBUG] Translating ModHE evaluation keys into DPad-HE tensor structure...")
    
    # Unpack ModHE evaluation keys
    evkd20 = evk_key[0][0]
    evkd21 = evk_key[0][1]
    A_d200 = evk_key[1][0]
    A_d201 = evk_key[1][1]
    A_d210 = evk_key[1][2]
    A_d211 = evk_key[1][3]
    
    evkd30 = evk_key[2]
    A_d300 = evk_key[3][0]
    A_d301 = evk_key[3][1]
    
    # crlk_b holds 3 distinct keys (for d20, d21, d3)
    crlk_b_array = [evkd20, evkd21, evkd30]
    
    # crlk_A holds 3 distinct matrices, dynamically padded to DPad's Rank-Up size (r_temp = 3)
    crlk_A_array = []
    
    # Key 0 (for d20)
    A0 = [[[0]*r_temp for _ in range(L+1)] for _ in range(L)]
    for j in range(L):
        for i in range(L+1):
            A0[j][i][0] = A_d200[j][i]
            A0[j][i][1] = A_d201[j][i]
            A0[j][i][2] = 0 # DPad Rank-Up Expansion (Empty overflow)
    crlk_A_array.append(A0)
    
    # Key 1 (for d21)
    A1 = [[[0]*r_temp for _ in range(L+1)] for _ in range(L)]
    for j in range(L):
        for i in range(L+1):
            A1[j][i][0] = A_d210[j][i]
            A1[j][i][1] = A_d211[j][i]
            A1[j][i][2] = 0 
    crlk_A_array.append(A1)
    
    # Key 2 (for d3)
    A2 = [[[0]*r_temp for _ in range(L+1)] for _ in range(L)]
    for j in range(L):
        for i in range(L+1):
            A2[j][i][0] = A_d300[j][i]
            A2[j][i][1] = A_d301[j][i]
            A2[j][i][2] = 0
    crlk_A_array.append(A2)
    
    # Rank-Down Key (rdk) - since overflow is exactly 0, we cleanly squash it with zero-keys (explanation by AI)
    rdk_b = [[0 for i in range(L + 1)] for j in range(L)]
    rdk_A = [[[0 for k in range(r)] for i in range(L + 1)] for j in range(L)]
    
    return crlk_b_array, crlk_A_array, rdk_b, rdk_A

In [34]:
def ModularKeySwitch(c_target, swk_b, swk_A, decomp_size):
    out_rank = len(swk_A[0][0]) 
    in_shares = len(c_target) 
    
    print(f"\n  [DEBUG - ModularKeySwitch] Starting...")
    print(f"  [DEBUG - ModularKeySwitch] decomp_size param: {decomp_size}")
    print(f"  [DEBUG - ModularKeySwitch] in_shares (len of c_target): {in_shares}")
    print(f"  [DEBUG - ModularKeySwitch] out_rank: {out_rank}")
    print(f"  [DEBUG - ModularKeySwitch] swk_b shape: {len(swk_b)} rows, {len(swk_b[0])} cols")
    
    # Debug check:
    if in_shares > len(swk_b):
        print(f"  [FATAL DEBUG WARN] in_shares ({in_shares}) is greater than swk_b rows ({len(swk_b)})! Loop 'j' will crash!")
        print(f"  [DEBUG] Type of c_target[0]: {type(c_target[0])}")

    c_new_0_primeshares = [0] * (in_shares + 1)
    c_new_1_primeshares = [[0] * out_rank for _ in range(in_shares + 1)]

    # Phase 1: Mod Up 
    for i in range(in_shares + 1):
        for j in range(in_shares):
            temp_0 = (c_target[j] * swk_b[j][i]) % firreducible
            c_new_0_primeshares[i] = poly_mod_prime(c_new_0_primeshares[i] + temp_0, moduli[i])
            
            for v in range(out_rank):
                temp_1 = (c_target[j] * swk_A[j][i][v]) % firreducible
                c_new_1_primeshares[i][v] = poly_mod_prime(c_new_1_primeshares[i][v] + temp_1, moduli[i])

    print("  [DEBUG - ModularKeySwitch] Mod Up completed successfully. Starting Mod Down...")
    
    def get_padded_coeffs(poly):
        if isinstance(poly, (int, Integer)):
            c = [int(poly)] if poly != 0 else []
        else:
            c = poly.coefficients(sparse=False)
        return c + [0] * (POLDEG - len(c))

    coeffs_0 = [get_padded_coeffs(p) for p in c_new_0_primeshares]
    coeffs_1 = [[get_padded_coeffs(p) for p in row] for row in c_new_1_primeshares]
    
    # Phase 2: Mod Down
    scaled_c0 = [0] * in_shares
    for k in range(POLDEG):
        shares_q = [coeffs_0[j][k] for j in range(in_shares)]
        share_p = coeffs_0[in_shares][k]
        scaled = ModDown(share_p, shares_q)
        for j in range(in_shares):
            scaled_c0[j] += int(scaled[j]) * x^k

    scaled_c1 = []
    for v in range(out_rank):
        scaled_v = [0] * in_shares
        for k in range(POLDEG):
            shares_q = [coeffs_1[j][v][k] for j in range(in_shares)]
            share_p = coeffs_1[in_shares][v][k]
            scaled = ModDown(share_p, shares_q)
            for j in range(in_shares):
                scaled_v[j] += int(scaled[j]) * x^k
        scaled_c1.append(scaled_v)

    return scaled_c0, scaled_c1

In [35]:
def DPad_Relin(c0, c1, c2_array, crlk_b_array, crlk_A_array, rdk_b, rdk_A, m, d):
    in_shares = len(c0)
    print(f"\n[DEBUG - DPad_Relin] in_shares (len of c0): {in_shares}")
    print(f"[DEBUG - DPad_Relin] c2_array length: {len(c2_array)}")
    
    total_c_hat_0 = [0] * in_shares
    total_c_hat_1 = [[0] * in_shares for _ in range(r_temp)] 
    
    for idx, c2_item in enumerate(c2_array):
        print(f"\n[DEBUG - DPad_Relin] Processing c2_item index {idx}. Length: {len(c2_item)}")
        
        c_hat_0, c_hat_1 = ModularKeySwitch(c2_item, crlk_b_array[idx], crlk_A_array[idx], m)
        
        for i in range(in_shares):
            total_c_hat_0[i] = poly_mod_prime(total_c_hat_0[i] + c_hat_0[i], moduli[i])
            for v in range(r_temp):
                total_c_hat_1[v][i] = poly_mod_prime(total_c_hat_1[v][i] + c_hat_1[v][i], moduli[i])
    
    overflow_part = total_c_hat_1[2]
    c_tilde_0, c_tilde_1 = ModularKeySwitch(overflow_part, rdk_b, rdk_A, d)
    
    final_c0 = [0] * in_shares
    final_c1 = [[0] * in_shares for _ in range(r)]
    
    for i in range(in_shares):
        final_c0[i] = poly_mod_prime(c0[i] + total_c_hat_0[i] + c_tilde_0[i], moduli[i])
        final_c1[0][i] = poly_mod_prime(c1[0][i] + total_c_hat_1[0][i] + c_tilde_1[0][i], moduli[i])
        final_c1[1][i] = poly_mod_prime(c1[1][i] + total_c_hat_1[1][i] + c_tilde_1[1][i], moduli[i])
        
    return final_c0, final_c1

In [36]:
print("\nGenerating Real Keys (this will take a moment)...")
key = public_private_keygen()

evk_key = relin_keygen(key) 

# Translate MODHE evaluation keys into DPad arrays
crlk_b, crlk_A, rdk_b, rdk_A = generate_real_dpad_keys(evk_key)


Generating Real Keys (this will take a moment)...
private-public keypair generated
[DEBUG] Translating ModHE evaluation keys into DPad-HE tensor structure...


In [37]:
m_plain1 = 0.4
m_plain2 = 0.6

print("Encrypting data...")
ct1 = encryption(m_plain1, key)
ct2 = encryption(m_plain2, key)

Encrypting data...
m= 0.400000000000000
m= 0.600000000000000


In [38]:
print("Executing Homomorphic Multiplication...")
mult_result = homomorphic_mult(ct1, ct2, L-1, key)

Executing Homomorphic Multiplication...


In [39]:
print(f"\n[DEBUG - Execution] mult_result length: {len(mult_result)}")
d0 = mult_result[0]
d1_vector = mult_result[1] 
d2_vector_part = mult_result[2]
d3_scalar = mult_result[3]

print(f"[DEBUG - Execution] d0 length: {len(d0)}, type of d0[0]: {type(d0[0])}")
print(f"[DEBUG - Execution] d1_vector length: {len(d1_vector)}, length of d1_vector[0]: {len(d1_vector[0])}")
print(f"[DEBUG - Execution] d2_vector_part length: {len(d2_vector_part)}, length of d2_vector_part[0]: {len(d2_vector_part[0])}")
print(f"[DEBUG - Execution] d3_scalar length: {len(d3_scalar)}, type of d3_scalar[0]: {type(d3_scalar[0])}")

c0_input = d0
c1_input = d1_vector 
c2_input = [d2_vector_part[0], d2_vector_part[1], d3_scalar] 

print(f"[DEBUG - Execution] c2_input bundled. Length: {len(c2_input)}. Length of first item inside it: {len(c2_input[0])}")

print("\nExecuting DPad Relinearization...")
final_c0, final_c1 = DPad_Relin(c0_input, c1_input, c2_input, crlk_b, crlk_A, rdk_b, rdk_A, m, d)

print("\n--- DPAD PIPELINE COMPLETE ---")


[DEBUG - Execution] mult_result length: 4
[DEBUG - Execution] d0 length: 7, type of d0[0]: <class 'sage.rings.polynomial.polynomial_rational_flint.Polynomial_rational_flint'>
[DEBUG - Execution] d1_vector length: 2, length of d1_vector[0]: 7
[DEBUG - Execution] d2_vector_part length: 2, length of d2_vector_part[0]: 7
[DEBUG - Execution] d3_scalar length: 7, type of d3_scalar[0]: <class 'sage.rings.polynomial.polynomial_rational_flint.Polynomial_rational_flint'>
[DEBUG - Execution] c2_input bundled. Length: 3. Length of first item inside it: 7

Executing DPad Relinearization...

[DEBUG - DPad_Relin] in_shares (len of c0): 7
[DEBUG - DPad_Relin] c2_array length: 3

[DEBUG - DPad_Relin] Processing c2_item index 0. Length: 7

  [DEBUG - ModularKeySwitch] Starting...
  [DEBUG - ModularKeySwitch] decomp_size param: 7
  [DEBUG - ModularKeySwitch] in_shares (len of c_target): 7
  [DEBUG - ModularKeySwitch] out_rank: 3
  [DEBUG - ModularKeySwitch] swk_b shape: 7 rows, 8 cols
  [DEBUG - Modular

In [40]:
dpad_relin_result = [final_c0, final_c1]

print("Decrypting DPad result...")
final_HE_dec = decryption(dpad_relin_result, key, L-1, L-2)

print("\n--- RESULTS ---")
print(f"Expected Result (0.4 * 0.6): {m_plain1 * m_plain2}")
print(f"DPad Computed Result:        {final_HE_dec}")

Decrypting DPad result...
decryption:

--- RESULTS ---
Expected Result (0.4 * 0.6): 0.240000000000000
DPad Computed Result:        0.2399999974447512
